In [ ]:
!git clone https://github.com/Kaihua-Chen/diffusion-vas
%cd diffusion-vas

%mkdir checkpoints
%cd checkpoints
!git lfs install
!git clone https://huggingface.co/kaihuac/diffusion-vas-amodal-segmentation
!git clone https://huggingface.co/kaihuac/diffusion-vas-content-completion

!wget -O depth_anything_v2_vitl.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth?download=true
%cd ../..


Cloning into 'diffusion-vas'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 462 (delta 29), reused 28 (delta 28), pack-reused 427 (from 1)
Receiving objects: 100% (462/462), 102.79 MiB | 63.11 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/diffusion-vas
/content/diffusion-vas/checkpoints
Updated git hooks.
Git LFS initialized.
Cloning into 'diffusion-vas-amodal-segmentation'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 11), reused 0 (delta 0), pack-reused 4 (from 1)
Unpacking objects: 100% (48/48), 28.43 KiB | 2.58 MiB/s, done.
Filtering content: 100% (3/3), 3.03 GiB | 42.81 MiB/s, done.
Encountered 1 file(s) that may not have been copied correctly on Windows:
	unet/diffusion_pytorch_model.safetensors

See: `git lfs help smudge` for more details.
Cloning int

In [ ]:
!tar -xvf ff5da6d6ecae486bb294aeaf5ee8f8a1.tar.gz

Streaming output truncated to the last 5000 lines.
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00019.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00000.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00017.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00010.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00014.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00013.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00005.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00019.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00009.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00007.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/depth_00001.tiff
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/rgba_00021.png
ff5da6d6ecae486bb294aeaf5ee8f8a1/camera_0001/obj_0002/segmentation_00000.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
from PIL import Image

def copy_demo_format(
    root: Path,
    camera_index: int = 0,
    object_id: int = 1,
    target_root: Path = None
) -> Path:
    cam_folder = root / f"camera_{camera_index:04d}"

    if not cam_folder.exists():
        raise FileNotFoundError(f"Camera folder {cam_folder} not found.")

    if target_root is None:
        target_root = root

    # Create target structure
    new_folder = target_root / f"copy_cam_{camera_index}_obj_{object_id}"
    rgba_target = new_folder / "rgbs"
    seg_target = new_folder / "masks"
    rgba_target.mkdir(parents=True, exist_ok=True)
    seg_target.mkdir(parents=True, exist_ok=True)

    # Collect and sort source files
    rgba_files = sorted(cam_folder.glob("rgba_*.png"))
    seg_files = sorted(cam_folder.glob("segmentation_*.png"))

    assert len(rgba_files) == len(seg_files), "Mismatch between RGB and segmentation files."

    limit = 25
    count = min(limit, len(rgba_files))

    for i in range(count):
        # Copy RGBA image
        rgba_dest = rgba_target / f"rgba_{i:05d}.png"
        shutil.copy(rgba_files[i], rgba_dest)

        # Process and save mask for the correct object ID
        seg_img = np.array(Image.open(seg_files[i]))
        binary_mask = (seg_img == object_id).astype(np.uint8) * 255
        mask_img = Image.fromarray(binary_mask)
        mask_dest = seg_target / f"segmentation_{i:05d}.png"
        mask_img.save(mask_dest)

    # Pad RGBA and masks if fewer than 25 frames
    if count < limit:
        last_rgba = rgba_files[count - 1]
        last_seg = np.array(Image.open(seg_files[count - 1]))
        last_mask = (last_seg == object_id).astype(np.uint8) * 255
        last_mask_img = Image.fromarray(last_mask)

        for i in range(count, limit):
            shutil.copy(last_rgba, rgba_target / f"rgba_{i:05d}.png")
            last_mask_img.save(seg_target / f"segmentation_{i:05d}.png")

    return new_folder


In [ ]:
root_path = Path("/content/ff5da6d6ecae486bb294aeaf5ee8f8a1")
target_path = Path("/content/diffusion-vas/demo_data")
copy_demo_format(root_path,camera_index=0,object_id=10,target_root=target_path)

PosixPath('/content/diffusion-vas/demo_data/copy_cam_0_obj_10')

In [ ]:
%cd /content/diffusion-vas
!python demo.py --seq_name "copy_cam_0_obj_10"

/content/diffusion-vas
2025-07-18 20:11:56.905580: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-18 20:11:56.922077: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752869516.943085    2663 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752869516.949415    2663 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-18 20:11:56.970310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to 